# 8 · Práctica integradora — Informe de Análisis Exploratorio (EDA)


> **¿Qué es esta práctica?**
> Es un **EDA completo**, de punta a punta, siguiendo los 5 pasos del flujo visto en la teoría:
> **01 Definir preguntas → 02 Conocer los datos → 03 Clasificar variables → 04 Evaluar la calidad → 05 Explorar y analizar**.
> Usa **solo** técnicas vistas en los notebooks 0 a 5. El resultado final es un **informe**: gráficos + conclusiones escritas.

> **Dataset nuevo (pero conocido):** usamos la versión **original** del *Breast Cancer Wisconsin (Diagnostic)*. Las variables son las mismas que en `sample_dataset.csv`, pero:
> - los nombres cambian de formato: `mean radius` → `radius_mean`, `radius error` → `radius_se`, `worst radius` → `radius_worst`;
> - el diagnóstico está en la columna **`diagnosis`** como texto: **`M` = maligno**, **`B` = benigno**;
> - hay una columna `id` (identificador de la muestra).

> **Si el notebook se reparte entre dos equipos:** Parte A = secciones 0 a 5 (calidad, univariado, outliers) · Parte B = secciones 6 a 9 (relaciones, correlación, informe). La Parte B igual ejecuta las celdas de las secciones 0 y 2 para cargar los datos.

**Cómo trabajar:** igual que en las guías: 📘 concepto · 💻 ejecutar · 🧩 completar `COMPLETAR` · ✍️ responder. En las secciones finales hay menos código armado: la idea es que **ustedes decidan** qué mirar.

## ✅ Checklist de entrega
- 1. Pregunta y fuente de datos
- 2. Exploración inicial y calidad de datos (faltantes, duplicados, columnas inútiles)
- 3. Clasificación de variables y balance de clases
- 4. Análisis univariado: descriptivos e histogramas (regla de Rice)
- 5. Boxplots, RIC y outliers + decisión fundamentada
- 6. Pairplot / jointplot con `hue`
- 7. Correlación (Pearson y Spearman) + heatmap
- 8. Informe final: 3 a 5 conclusiones con evidencia + limitaciones

## 0) Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)   # mostrar todas las columnas al imprimir un DataFrame

## 1) Definir la pregunta y conocer la fuente (paso 01)

✍️ **Preguntas**
1. Escriban con sus palabras la **pregunta** que guía este análisis.
2. ¿Qué tipo de fuente es (primaria, secundaria o terciaria)? ¿Por qué importa saberlo?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 2) Carga, exploración inicial y calidad de datos (pasos 02 y 04 — notebooks 0 y 1)

In [ ]:
#@title 💻 Cargar el dataset (ejecutar sin modificar)
url = "https://huggingface.co/datasets/scikit-learn/breast-cancer-wisconsin/raw/main/breast_cancer.csv"
try:
    df = pd.read_csv(url)
    print("Dataset cargado desde la URL.")
except Exception as e:
    # Plan B por si falla internet: armamos la misma tabla desde scikit-learn
    from sklearn.datasets import load_breast_cancer
    bc = load_breast_cancer(as_frame=True)
    datos = bc.frame.drop(columns="target")
    nuevos = {}
    for c in datos.columns:
        if c.startswith("mean "):
            nuevos[c] = c[5:].replace(" ", "_") + "_mean"
        elif c.startswith("worst "):
            nuevos[c] = c[6:].replace(" ", "_") + "_worst"
        else:
            nuevos[c] = c.replace(" error", "").replace(" ", "_") + "_se"
    df = datos.rename(columns=nuevos)
    df.insert(0, "diagnosis", bc.frame["target"].map({0: "M", 1: "B"}))
    df.insert(0, "id", range(1, len(df) + 1))
    print("No se pudo leer la URL; se cargó la versión de scikit-learn.")
df.head()

🧩 Completen con lo visto en el notebook 0: **dimensiones** y **tipos de datos**.

In [ ]:
print("Filas y columnas:", df.COMPLETAR)     # pista: es un atributo, va sin paréntesis
df.COMPLETAR                                 # tipos de dato de cada columna

In [ ]:
# Faltantes por columna (de mayor a menor) y filas duplicadas
print("Faltantes por columna:")
print(df.isna().sum().sort_values(ascending=False).head(5))
print("\nFilas duplicadas:", df.duplicated().sum())

# Columnas completamente vacías (pasa a veces con CSV exportados con una coma de más)
vacias = df.columns[df.isna().all()].tolist()
print("Columnas 100% vacías:", vacias)

📘 **Columnas que no aportan al análisis.** El `id` es solo un identificador: no mide nada de la célula, así que no tiene sentido promediarlo, graficarlo ni correlacionarlo. Las columnas 100% vacías tampoco aportan. Las sacamos **documentando la decisión**.

In [ ]:
df = df.drop(columns=["id"] + vacias)
df.shape

✍️ **Diagnóstico de calidad de datos.** Comparen con `sample_dataset.csv` (notebooks 0 y 1):
1. ¿Hay faltantes? ¿Duplicados? ¿Tipos de dato incorrectos?
2. ¿Qué columnas eliminaron y por qué?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 3) Clasificar variables y balance de clases (paso 03)

In [ ]:
print(df["diagnosis"].value_counts())
print(df["diagnosis"].value_counts(normalize=True).round(3))

df["diagnosis"].value_counts().plot(kind="bar", title="diagnosis (M = maligno, B = benigno)", rot=0)
plt.show()

✍️ **Preguntas**
1. Clasifiquen `diagnosis` y `radius_mean` según la lámina *"Clasificación de variables"* (categórica nominal/ordinal, numérica discreta/continua).
2. ¿Las clases están balanceadas?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 4) Análisis univariado (paso 05 — notebooks 1 y 2)

💻 **Descriptivos.** Recuerden la lectura rápida del notebook 1: *count*, *media vs. mediana (50%)*, *max vs. 75%*, *escalas*.

In [ ]:
df.describe().T

✍️ **Preguntas**
1. ¿En qué variables la media es claramente mayor que la mediana? ¿Qué sugiere eso?
2. ¿Cuál es el orden de magnitud de `area_mean` vs. `smoothness_mean`? ¿Consideraría **estandarizar** en un futuro modelado?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

📘 **Histogramas con la regla de Rice:** $k = \lceil 2 \cdot n^{1/3} \rceil$

🧩 Completen la función y elijan **3 variables** para graficar (incluyan al menos `radius_mean` y `area_mean`).

In [ ]:
def rice_bins(n):
    return int(np.ceil(2 * n ** (COMPLETAR)))

bins = rice_bins(len(df))
print("Bins (Rice):", bins)

cols = ["radius_mean", "area_mean", COMPLETAR]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, c in zip(axes, cols):
    ax.hist(df[c].dropna(), bins=bins)
    ax.axvline(df[c].mean(), color="red", linestyle="--", label="media")
    ax.axvline(df[c].median(), color="green", linestyle="--", label="mediana")
    ax.set_title(f"{c} | skew = {df[c].skew():.2f}")
    ax.legend()
plt.show()

✍️ **Pregunta:** ¿Las distribuciones son simétricas o sesgadas? ¿Qué implicancia tiene para elegir entre media y mediana?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 5) Boxplots, RIC y outliers (paso 04/05 — notebook 3)

📘 Recordatorio: RIC = Q3 − Q1 · límites = Q1 − 1,5·RIC y Q3 + 1,5·RIC · outlier = valor fuera de esos límites.

In [ ]:
def outlier_stats(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < low) | (s > high)
    return pd.Series({"Q1": q1, "Q3": q3, "RIC": iqr, "lim_inf": low, "lim_sup": high, "outliers": int(mask.sum())})

# Tabla resumen para las 3 variables elegidas en la sección 4
pd.DataFrame({c: outlier_stats(df[c]) for c in cols}).T.round(3)

🧩 Completen para hacer un **boxplot por diagnóstico** de cada variable (pista: la columna que separa los grupos).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, c in zip(axes, cols):
    sns.boxplot(data=df, x=COMPLETAR, y=c, ax=ax)
    ax.set_title(c)
plt.show()

✍️ **Pregunta:** ¿Cuántos outliers tiene cada variable y a qué diagnóstico corresponden? En este contexto exploratorio, ¿corresponde **eliminarlos** o **mantenerlos**? Justifiquen.

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 6) Relaciones entre variables: pairplot y jointplot (notebook 4)

🧩 Elijan 4 o 5 variables (incluyan `radius_mean` y `area_mean`) y armen el pairplot con color por diagnóstico.

In [ ]:
subset = ["radius_mean", "area_mean", COMPLETAR, COMPLETAR]
sns.pairplot(df, vars=subset, hue=COMPLETAR, corner=True)
plt.show()

In [ ]:
sns.jointplot(data=df, x="perimeter_mean", y="area_mean", hue="diagnosis")
plt.show()

✍️ **Pregunta:** Comenten **dos relaciones** interesantes (dirección, fuerza, si es lineal o curva) y qué variable separa mejor los diagnósticos.

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 7) Correlación: Pearson vs. Spearman + heatmap (notebook 5)

📘 Para poder correlacionar el diagnóstico, lo pasamos a número. **Atención:** acá usamos **1 = maligno** (al revés que `target` en `sample_dataset.csv`), así que una correlación **positiva** indica asociación con **maligno**.

In [ ]:
df["maligno"] = (df["diagnosis"] == "M").astype(int)   # 1 = maligno, 0 = benigno
num_df = df.select_dtypes(include="number")

corr_p = num_df.corr(method="pearson")
corr_s = num_df.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
sns.heatmap(corr_p, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0]); axes[0].set_title("Pearson")
sns.heatmap(corr_s, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1]); axes[1].set_title("Spearman")
plt.tight_layout()
plt.show()

🧩 Ranking de variables más asociadas al diagnóstico con **los dos métodos**. Completen el nombre de la columna.

In [ ]:
ranking = pd.DataFrame({
    "pearson": corr_p[COMPLETAR],
    "spearman": corr_s[COMPLETAR],
}).drop("maligno").sort_values("pearson", ascending=False)
ranking.head(10).round(2)

✍️ **Preguntas**
1. Expliquen en 3–4 líneas **cuándo usar Pearson y cuándo Spearman**, y por qué en algunos pares cambia la magnitud.
2. ¿Qué variables son **redundantes** entre sí (correlación muy alta)? ¿Qué implicancia tiene?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 8) 📝 Informe final

Completen la tabla con **entre 3 y 5 conclusiones**. Cada conclusión necesita **evidencia** (un gráfico o un número de este notebook).

| # | Hallazgo | Evidencia (gráfico / número) | ¿Qué implica? |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

**Variables más informativas para diferenciar diagnósticos:** ______

**Limitaciones de este análisis** (lámina *"Cómo interpretar los resultados de un EDA"*: causalidad, calidad de datos, alcance exploratorio, contexto): ______

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 9) ⭐ (Opcional) Reporte automático (notebooks 6 y 7)

Las librerías **ydata-profiling** y **Sweetviz** generan en pocas líneas un informe con buena parte de lo que hicimos a mano. Úsenlo para **contrastar** sus conclusiones, no para reemplazarlas.

> Si Colab pide **reiniciar el entorno** después del `pip install`, reinicien y vuelvan a ejecutar desde el Setup.

In [ ]:
# !pip install ydata-profiling
# from ydata_profiling import ProfileReport
# reporte = ProfileReport(df[["diagnosis"] + cols], minimal=True)
# reporte.to_notebook_iframe()